In [2]:
import fastf1
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

fastf1.Cache.enable_cache("f1_cache")

In [3]:
q = fastf1.get_session(2024, "Canada", "Qualifying")
q.load()  # loads quali data (laps, results, etc.)

# Best qualifying lap per driver (seconds) + Team
q_laps = q.laps.dropna(subset=["LapTime"]).copy()
best_q = (
    q_laps.sort_values("LapTime")
          .groupby("Driver", as_index=False)
          .first()[["Driver", "Team", "LapTime"]]
          .rename(columns={"LapTime": "QualifyingTime_s"})
)
best_q["QualifyingTime_s"] = best_q["QualifyingTime_s"].dt.total_seconds()

core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.6.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO

In [4]:
r = fastf1.get_session(2024, "Canada", "R")
r.load()

laps = r.laps.dropna(subset=["LapTime"]).copy()
# Keep “proper” race laps: drop pit in/out laps (simple, robust rule)
if "PitInLap" in laps.columns and "PitOutLap" in laps.columns:
    laps = laps[~(laps["PitInLap"] | laps["PitOutLap"])].copy()

laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()

# Trim per-driver outliers (5th–95th percentile) to stabilize the target
def _trim(g):
    lo, hi = np.percentile(g["LapTime_s"], [5, 95])
    return g[(g["LapTime_s"] >= lo) & (g["LapTime_s"] <= hi)]

laps_trim = laps.groupby("Driver", group_keys=False).apply(_trim)

y_tbl = laps_trim.groupby("Driver", as_index=False)["LapTime_s"].mean()
y_tbl = y_tbl.rename(columns={"LapTime_s": "TargetMeanRaceLap_s"})

core           INFO 	Loading data for Canadian Grand Prix - Race [v3.6.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No c

In [5]:
train_df = best_q.merge(y_tbl, on="Driver", how="inner")

# Features: ONLY qualifying time (+ optional Team one-hot). No weather.
feature_cols_num = ["QualifyingTime_s"]
feature_cols_cat = ["Team"]  # optional but useful
X = train_df[feature_cols_num + feature_cols_cat].copy()
y = train_df["TargetMeanRaceLap_s"].copy()

In [6]:
num_tf = Pipeline([("imp", SimpleImputer(strategy="median"))])
cat_tf = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pre = ColumnTransformer([
    ("num", num_tf, feature_cols_num),
    ("cat", cat_tf, feature_cols_cat),
])

model = Pipeline([
    ("pre", pre),
    ("gbr", GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=3, random_state=42
    )),
])

# Small train/test split across drivers (it’s still one event, but fine for a demo)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=7)
model.fit(X_tr, y_tr)
y_hat = model.predict(X_te)
print(f"MAE on 2024 Canada (driver hold-out): {mean_absolute_error(y_te, y_hat):.3f} s")

MAE on 2024 Canada (driver hold-out): 0.457 s


In [7]:
q2025 = fastf1.get_session(2025, "Canada", "Q")  # "Q" or "Qualifying" both work
q2025.load()

# results has columns like: Abbreviation (driver code), TeamName, Q1, Q2, Q3 (strings)
res25 = q2025.results[["Abbreviation", "TeamName", "Q1", "Q2", "Q3"]].copy()

# convert Q1/Q2/Q3 to timedeltas, then seconds
for col in ["Q1", "Q2", "Q3"]:
    res25[col] = pd.to_timedelta(res25[col], errors="coerce")

res25["QualifyingTime_s"] = res25[["Q1","Q2","Q3"]].min(axis=1).dt.total_seconds()

inf_2025 = res25.rename(columns={"Abbreviation":"Driver", "TeamName":"Team"})[["Driver","Team","QualifyingTime_s"]]

# (Optional) filter to drivers you trained on (if needed)
# inf_2025 = inf_2025[inf_2025["Driver"].isin(train_df["Driver"].unique())]

# predict
X_2025 = inf_2025[feature_cols_num + feature_cols_cat].copy()  # same cols/order as training
pred_2025 = model.predict(X_2025)

out_2025 = inf_2025[["Driver","Team","QualifyingTime_s"]].copy()
out_2025["PredictedMeanRaceLap_s"] = pred_2025
out_2025 = out_2025.sort_values("PredictedMeanRaceLap_s").reset_index(drop=True)

print("\n=== Montreal 2025 — predicted race pace (lower is faster) ===")
print(out_2025)

core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.6.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO


=== Montreal 2025 — predicted race pace (lower is faster) ===
   Driver             Team  QualifyingTime_s  PredictedMeanRaceLap_s
0     PIA          McLaren            71.120               88.906729
1     NOR          McLaren            71.599               88.906729
2     ALO     Aston Martin            71.586               89.014137
3     VER  Red Bull Racing            71.059               89.022718
4     HAD     Racing Bulls            71.867               89.112829
5     ANT         Mercedes            71.391               89.143614
6     RUS         Mercedes            70.899               89.143614
7     HAM          Ferrari            71.526               89.423299
8     LEC          Ferrari            71.626               89.423299
9     BEA     Haas F1 Team            72.306               89.468491
10    OCO     Haas F1 Team            72.378               89.587690
11    COL           Alpine            72.142               89.609368
12    TSU  Red Bull Racing            72

In [9]:
race25 = fastf1.get_session(2025, "Canada", "R")
race25.load()

# Remove pit in/out laps and safety car laps
laps = race25.laps.dropna(subset=["LapTime"]).copy()
if "PitInLap" in laps.columns and "PitOutLap" in laps.columns:
    laps = laps[~(laps["PitInLap"] | laps["PitOutLap"])]

# Remove laps under VSC/SC if info available
if "TrackStatus" in laps.columns:
    laps = laps[~laps["TrackStatus"].astype(str).str.contains("4|5", na=False)]  # 4=SC, 5=VSC

laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()

# Compute actual median race pace
actual_pace = laps.groupby("Driver", as_index=False)["LapTime_s"].median()
actual_pace = actual_pace.rename(columns={"LapTime_s": "ActualMedianRaceLap_s"})

# Merge with prediction
compare = out_2025.merge(actual_pace, on="Driver", how="left")
compare["Error_s"] = compare["PredictedMeanRaceLap_s"] - compare["ActualMedianRaceLap_s"]

# Sort by actual pace
compare = compare.sort_values("ActualMedianRaceLap_s").reset_index(drop=True)

print(compare)

core           INFO 	Loading data for Canadian Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '1', '12', '81', '16', '44', '14', '27', '31', '55', '87', '22', '43', '5', '10', '6', '18', '4', '30', '23']


   Driver             Team  QualifyingTime_s  PredictedMeanRaceLap_s  \
0     RUS         Mercedes            70.899               89.143614   
1     ANT         Mercedes            71.391               89.143614   
2     PIA          McLaren            71.120               88.906729   
3     NOR          McLaren            71.599               88.906729   
4     VER  Red Bull Racing            71.059               89.022718   
5     LEC          Ferrari            71.626               89.423299   
6     HAM          Ferrari            71.526               89.423299   
7     ALO     Aston Martin            71.586               89.014137   
8     HUL      Kick Sauber            72.183               89.798932   
9     BEA     Haas F1 Team            72.306               89.468491   
10    OCO     Haas F1 Team            72.378               89.587690   
11    SAI         Williams            72.398               90.928530   
12    HAD     Racing Bulls            71.867               89.11

/tmp/ipykernel_17516/1241460953.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps["LapTime_s"] = laps["LapTime"].dt.total_seconds()
